# Projeto de Modelagem Estatística — Temperatura de Belém

**Dataset:** Global Land Temperatures by City
**Recorte:** Belém, Pará, Brasil
**Período válido:** 1845–2013

## Pergunta de pesquisa
Como a temperatura média anual de Belém se relaciona com o passar dos anos e qual é a capacidade de um modelo de regressão linear simples representar essa tendência?

## Objetivos
- Explorar a série histórica.
- Tratar valores ausentes.
- Criar variáveis de ano e mês.
- Aplicar estatística descritiva.
- Ajustar regressão linear.
- Testar hipóteses e construir intervalo de confiança.
- Avaliar o modelo em um período posterior.
- Extrapolar a tendência para 2014–2020.


In [ ]:
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

with zipfile.ZipFile("GlobalLandTemperaturesByCity.csv.zip") as z:
    nome = z.namelist()[0]
    with z.open(nome) as f:
        partes = []
        for chunk in pd.read_csv(f, chunksize=300000):
            parte = chunk[(chunk["Country"] == "Brazil") & (chunk["City"] == "Belém")].copy()
            if not parte.empty:
                partes.append(parte)
belem = pd.concat(partes, ignore_index=True)
belem["dt"] = pd.to_datetime(belem["dt"])
belem["year"] = belem["dt"].dt.year
belem["month"] = belem["dt"].dt.month
belem.shape

## Tratamento dos dados

Registros sem temperatura média são excluídos da análise principal. A temperatura média anual é calculada pela média dos registros mensais disponíveis em cada ano, sem imputação artificial dos valores ausentes.

In [ ]:
belem.isna().sum()

In [ ]:
belem_valid = belem.dropna(subset=["AverageTemperature"]).copy()
anual = belem_valid.groupby("year", as_index=False).agg(AverageTemperature=("AverageTemperature","mean"), AverageTemperatureUncertainty=("AverageTemperatureUncertainty","mean"))
mensal = belem_valid.groupby("month", as_index=False).agg(AverageTemperature=("AverageTemperature","mean"), AverageTemperatureUncertainty=("AverageTemperatureUncertainty","mean"))
anual.shape, anual.head(), anual.tail()

In [ ]:
anual["AverageTemperature"].describe()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(anual["year"], anual["AverageTemperature"])
plt.xlabel("Ano")
plt.ylabel("Temperatura média anual (°C)")
plt.title("Temperatura média anual de Belém")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(mensal["month"], mensal["AverageTemperature"], marker="o")
plt.xlabel("Mês")
plt.ylabel("Temperatura média (°C)")
plt.title("Temperatura média histórica por mês em Belém")
plt.xticks(range(1,13))
plt.grid(True)
plt.show()

## Regressão linear

Modelo: Temperatura = β₀ + β₁ × Ano + ε

**H₀:** β₁ = 0

**H₁:** β₁ ≠ 0

Nível de significância: 5%.

In [ ]:
reg = stats.linregress(anual["year"], anual["AverageTemperature"])
pd.DataFrame({"Métrica":["Inclinação","Intercepto","R²","p-valor","Erro padrão"],"Valor":[reg.slope,reg.intercept,reg.rvalue**2,reg.pvalue,reg.stderr]})

In [ ]:
tcrit = stats.t.ppf(0.975, len(anual)-2)
ic_inclinacao = (reg.slope-tcrit*reg.stderr, reg.slope+tcrit*reg.stderr)
ic_inclinacao

In [ ]:
plt.figure(figsize=(12,5))
plt.scatter(anual["year"], anual["AverageTemperature"], s=20)
plt.plot(anual["year"], reg.intercept+reg.slope*anual["year"], linewidth=2)
plt.xlabel("Ano")
plt.ylabel("Temperatura média anual (°C)")
plt.title("Regressão linear — Temperatura de Belém × Ano")
plt.grid(True)
plt.show()

## Teste t de Welch entre períodos

Comparamos as médias anuais de 1845–1894 e 1964–2013.

**H₀:** as médias são iguais.

**H₁:** as médias são diferentes.

In [ ]:
periodo_inicial = anual[anual["year"].between(1845,1894)]["AverageTemperature"]
periodo_final = anual[anual["year"].between(1964,2013)]["AverageTemperature"]
teste_periodos = stats.ttest_ind(periodo_final, periodo_inicial, equal_var=False)
pd.DataFrame({"Métrica":["Média 1845–1894","Média 1964–2013","Estatística t","p-valor"],"Valor":[periodo_inicial.mean(),periodo_final.mean(),teste_periodos.statistic,teste_periodos.pvalue]})

## Avaliação temporal

O modelo é treinado até 1999 e avaliado nos anos 2000–2013. São calculados RMSE e MAE para verificar o quanto a tendência linear representa as observações posteriores.

In [ ]:
treino = anual[anual["year"] <= 1999]
teste = anual[anual["year"] >= 2000]
reg_treino = stats.linregress(treino["year"], treino["AverageTemperature"])
previsoes = reg_treino.intercept + reg_treino.slope*teste["year"]
rmse = np.sqrt(np.mean((teste["AverageTemperature"]-previsoes)**2))
mae = np.mean(np.abs(teste["AverageTemperature"]-previsoes))
pd.DataFrame({"Métrica":["RMSE","MAE","Anos de teste"],"Valor":[rmse,mae,len(teste)]})

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(treino["year"], treino["AverageTemperature"], label="Treino")
plt.plot(teste["year"], teste["AverageTemperature"], label="Observado")
plt.plot(teste["year"], previsoes, label="Previsão")
plt.xlabel("Ano")
plt.ylabel("Temperatura média anual (°C)")
plt.title("Avaliação temporal da regressão")
plt.legend()
plt.grid(True)
plt.show()

## Previsão de tendência

O modelo final usa todos os anos disponíveis e extrapola a tendência para 2014–2020. Esses valores não devem ser interpretados como previsão climática completa, pois o modelo utiliza somente o ano.

In [ ]:
reg_final = stats.linregress(anual["year"], anual["AverageTemperature"])
anos_futuros = pd.DataFrame({"year":range(2014,2021)})
anos_futuros["predicted_temperature"] = reg_final.intercept + reg_final.slope*anos_futuros["year"]
anos_futuros

## Conclusão

A análise combina estatística descritiva, tratamento de dados, regressão linear, teste de hipóteses, intervalo de confiança, avaliação temporal e previsão. A principal conclusão estatística deve ser apresentada junto das limitações do modelo e da natureza de extrapolação das previsões.